# ML-07b — Identifiants sémantiques : recommander un item que personne n'a noté

Ce carnet prolonge [ML-07-Recommendation-Python](ML-07-Recommendation-Python.ipynb). La factorisation matricielle y prédit des notes à partir de la **matrice d'interactions** : un item que personne n'a encore noté n'a aucune coordonnée latente, donc aucune chance d'être recommandé. C'est la faiblesse structurelle du filtrage collaboratif, et c'est précisément ce que la génération de candidats par le contenu cherche à couvrir.

Les recommandateurs génératifs récents franchissent un pas de plus : ils **tokenisent chaque item en un identifiant sémantique** — une suite de codes obtenue en quantifiant un plongement de son contenu — puis apprennent à générer ou à ranger ces identifiants. Le rapport technique Sona (équipe Yandex Music, [arXiv:2608.11015](https://arxiv.org/abs/2608.11015)) documente ce dispositif en production : un encodeur multimodal figé produit le plongement, un objectif contrastif collaboratif l'ajuste, une quantification résiduelle à plusieurs niveaux le compresse en tuples de codes.

Le carnet isole la question mesurable, et rien d'autre : **que gagne-t-on, et que perd-on, à ranger les items dans un espace d'identifiants quantifiés plutôt que directement dans l'espace de contenu ?** La réponse est mesurée, contre des baselines qui ne sont pas des hommes de paille.

**Crédit** : le paradigme et son vocabulaire (identifiant sémantique, quantification résiduelle à plusieurs niveaux, compression d'historique) viennent du rapport Sona. Aucune valeur du papier n'est recopiée ici : tout ce qui est chiffré dans ce carnet est calculé par ses propres cellules.

## Ce que le carnet mesure, en une phrase

Trois représentations d'items — populaire, collaborative, contenu, quantifiée — sont confrontées sur **deux populations de cible distinctes** : les items chauds (présents dans les interactions d'entraînement) et les items froids (présents au catalogue, absents de l'entraînement). Le résultat intéresse les deux, et le désaccord entre les deux est le contenu pédagogique.

In [1]:
import math
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

RNG = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)

print(f"numpy {np.__version__} | pandas {pd.__version__} | graine 42")

numpy 2.3.5 | pandas 3.0.2 | graine 42


### Pourquoi un corpus contrôlé

Un jeu de données réel ne dirait pas *pourquoi* un modèle gagne : les facteurs — qualité du contenu, densité des interactions, part d'items froids — y sont confondus. On génère donc un corpus dont la **vérité terrain est connue puis cachée aux algorithmes**, exactement comme [ML-08-Clustering-Python](ML-08-Clustering-Python.ipynb) génère sa vérité terrain par un mélange gaussien avant de la retirer.

La structure générative est volontairement simple et déclarée : chaque item appartient à un mélange de quelques styles latents et porte des étiquettes textuelles tirées du vocabulaire de ces styles ; chaque utilisateur a un goût, c'est-à-dire une distribution sur les mêmes styles. Les interactions sont ensuite tirées selon l'accord goût-contenu, pondéré par une popularité log-normale. **Le contenu est donc informatif par construction** — c'est la condition pour que la question posée ait un sens ; un corpus où le contenu serait du bruit ne mesurerait que le bruit.

In [2]:
STYLES = {
    "rock":        ["guitare", "distorsion", "riff", "batterie", "power", "anthem", "grunge", "punk", "solo", "ampli", "refrain", "indie"],
    "electro":     ["synthé", "basse", "bpm", "techno", "house", "sequencer", "808", "club", "loop", "modulaire", "drop", "acid"],
    "jazz":        ["saxophone", "improvisation", "swing", "contrebasse", "cuivres", "bop", "standard", "piano", "trio", "balade", "blue", "note"],
    "classique":   ["orchestre", "cordes", "sonate", "symphonie", "quatuor", "baroque", "chœur", "conducteur", "opus", "adagio", "fugue", "prélude"],
    "rap":         ["flow", "beat", "punchline", "sample", "boom", "bap", "couplet", "egotrip", "prod", "street", "mixtape", "clash"],
    "pop":         ["refrain", "hook", "radio", "mélodie", "chorus", "hit", "single", "voix", "production", "catchy", "estival", "dance"],
    "folk":        ["acoustique", "guitare", "harmonica", "ballade", "conte", "racines", "campagne", "voix", "cordes", "americana", "bluegrass", "nylon"],
    "metal":       ["double", "growl", "blast", "riff", "downpicking", "heavy", "thrash", "doom", "breakdown", "tremolo", "forge", "growler"],
    "reggae":      ["skank", "dub", "roots", "basse", "one", "drop", "toasting", "rocksteady", "echo", "delay", "sound", "system"],
    "ambient":     ["nappe", "drone", "texture", "réverbération", "minimal", "modulaire", "paysage", "lent", "espace", "calme", "field", "recording"],
    "world":       ["percussions", "polyrythmie", "traditionnel", "kora", "tabla", "gamelan", "chant", "modes", "acoustique", "rituel", "fête", "danse"],
    "chanson":     ["texte", "interprète", "couplet", "refrain", "piano", "voix", "auteur", "storytelling", "mélodie", "scène", "émotion", "français"],
}
STYLE_NAMES = list(STYLES)
ERAS = ["60s", "70s", "80s", "90s", "00s", "10s", "20s"]
FORMATS = ["single", "album", "live", "remix", "instrumental"]

N_ITEMS, N_USERS = 2000, 800
N_INTER_PER_USER = 40
COLD_FRACTION = 0.15

# --- items : melange de styles + etiquettes textuelles tirees de ces styles
item_mix = RNG.dirichlet(np.full(len(STYLE_NAMES), 0.25), size=N_ITEMS)
item_primary = item_mix.argmax(axis=1)
item_popularity = RNG.lognormal(mean=0.0, sigma=0.6, size=N_ITEMS)

metadata = []
for i in range(N_ITEMS):
    order = np.argsort(-item_mix[i])
    words = []
    for st in order[:2]:
        pool = STYLES[STYLE_NAMES[st]]
        k = 4 if st == order[0] else 2
        words += list(RNG.choice(pool, size=k, replace=False))
    words.append(ERAS[RNG.integers(len(ERAS))])
    words.append(FORMATS[RNG.integers(len(FORMATS))])
    metadata.append(" ".join(words))

# --- utilisateurs : gout sur les memes styles
user_pref = RNG.dirichlet(np.full(len(STYLE_NAMES), 0.35), size=N_USERS)

# --- interactions : accord gout-contenu pondere par la popularite
interactions = []
for u in range(N_USERS):
    affinity = user_pref[u] @ item_mix.T                      # (n_items,)
    weight = affinity * item_popularity
    prob = weight / weight.sum()
    chosen = RNG.choice(N_ITEMS, size=N_INTER_PER_USER, replace=False, p=prob)
    for it in chosen:
        interactions.append((u, int(it)))
interactions = np.array(interactions)
print(f"catalogue : {N_ITEMS} items | utilisateurs : {N_USERS} | interactions : {len(interactions)}")
print(f"vocabulaire d'etiquettes : {len({w for v in STYLES.values() for w in v} | set(ERAS) | set(FORMATS))} mots")
print(f"densite de la matrice : {len(interactions) / (N_ITEMS * N_USERS):.4f}")
print()
print("Trois exemples de metadonnees :")
for i in range(3):
    print(f"  item {i:4d} | styles dominants {STYLE_NAMES[item_mix[i].argmax()]:>9} "
          f"{STYLE_NAMES[np.argsort(-item_mix[i])[1]]:>9} | {metadata[i]}")

catalogue : 2000 items | utilisateurs : 800 | interactions : 32000
vocabulaire d'etiquettes : 141 mots
densite de la matrice : 0.0200

Trois exemples de metadonnees :
  item    0 | styles dominants     world   electro | percussions danse gamelan acoustique basse bpm 80s single
  item    1 | styles dominants    reggae   chanson | delay rocksteady one system émotion refrain 10s instrumental
  item    2 | styles dominants   ambient      rock | recording field minimal modulaire solo guitare 80s remix


Lecture de la sortie : le catalogue est **dense en contenu** (chaque item porte des étiquettes informatives) et **creux en interactions** (moins de trois pour cent des cases de la matrice). Cette asymétrie est la condition même du problème de démarrage à froid : le contenu voit tous les items, la matrice n'en voit qu'une partie.

L'étape suivante fabrique les deux populations de cible. Les items froids sont retirés **entièrement** de l'entraînement — aucune interaction ne les représente — mais leurs interactions réelles servent de cible de test. C'est le protocole honnête du démarrage à froid : l'item existe au catalogue et intéresse des utilisateurs, il manque seulement à la matrice.

In [3]:
n_items_cold = int(round(COLD_FRACTION * N_ITEMS))
cold_items = RNG.choice(N_ITEMS, size=n_items_cold, replace=False)
is_cold = np.zeros(N_ITEMS, dtype=bool)
is_cold[cold_items] = True

# interactions d'entrainement : tout sauf les items froids
train_pairs = interactions[~is_cold[interactions[:, 1]]]
cold_pairs = interactions[is_cold[interactions[:, 1]]]

# cible "froide" : une interaction froide par utilisateur qui en a au moins une
cold_target = {}
for u, i in cold_pairs:
    cold_target.setdefault(int(u), int(i))

# cible "chaude" : derniere interaction chaude de l'utilisateur (leave-last-out)
warm_by_user = {}
for u, i in train_pairs:
    warm_by_user.setdefault(int(u), []).append(int(i))
warm_target = {}
train_pairs_final = []
for u, items_u in warm_by_user.items():
    warm_target[u] = items_u[-1]
    for i in items_u[:-1]:
        train_pairs_final.append((u, i))
train_pairs_final = np.array(train_pairs_final)

print(f"items froids : {n_items_cold} / {N_ITEMS} ({n_items_cold / N_ITEMS:.0%})")
print(f"interactions d'entrainement : {len(train_pairs_final)}")
print(f"cibles chaudes (leave-last-out) : {len(warm_target)} utilisateurs")
print(f"cibles froides (items hors entrainement) : {len(cold_target)} utilisateurs")
print(f"interactions froides retirees de l'entrainement : {len(cold_pairs)}")

items froids : 300 / 2000 (15%)
interactions d'entrainement : 26222
cibles chaudes (leave-last-out) : 800 utilisateurs
cibles froides (items hors entrainement) : 798 utilisateurs
interactions froides retirees de l'entrainement : 4978


Chaque utilisateur contribue donc **deux fois** à l'évaluation, avec deux cibles de natures différentes et le même univers de candidats. Une seule matrice d'interactions d'entraînement sert aux deux protocoles : c'est ce qui rend les chiffres comparables.

## Le plongement de contenu

Le premier étage est celui que Sona confie à un encodeur multimodal figé. Ici le contenu est du texte : on le vectorise par sac de mots hachés, pondéré par la fréquence inverse du document, puis réduit par décomposition en valeurs singulières. Les étiquettes de style sont ce qui doit survivre à la réduction — la cellule suivante le vérifie en clair plutôt que de le supposer.

In [4]:
VOCAB = sorted({w for v in STYLES.values() for w in v} | set(ERAS) | set(FORMATS))
vocab_index = {w: j for j, w in enumerate(VOCAB)}

def bag_of_words(text):
    v = np.zeros(len(VOCAB))
    for w in text.split():
        if w in vocab_index:
            v[vocab_index[w]] += 1.0
    return v

X_text = np.vstack([bag_of_words(t) for t in metadata])
df = (X_text > 0).sum(axis=0)
idf = np.log((1 + N_ITEMS) / (1 + df)) + 1.0
X_tfidf = (X_text > 0) * idf * (1 + np.log(np.clip(X_text, 1, None)))
X_tfidf /= np.linalg.norm(X_tfidf, axis=1, keepdims=True)

EMB_DIM = 32
svd = TruncatedSVD(n_components=EMB_DIM, random_state=42)
E = svd.fit_transform(X_tfidf)
E /= np.linalg.norm(E, axis=1, keepdims=True)

print(f"plongement de contenu : {E.shape[0]} items x {E.shape[1]} dimensions")
print(f"variance expliquee cumulee : {svd.explained_variance_ratio_.sum():.3f}")
print()
print("Controle : les items partageant le style dominant se ressemblent-ils dans l'espace reduit ?")
sims_intra, sims_inter = [], []
for i in range(300):
    same = np.where(item_primary == item_primary[i])[0]
    same = same[same != i]
    other = np.where(item_primary != item_primary[i])[0]
    sims_intra.append(float(E[i] @ E[RNG.choice(same)].T))
    sims_inter.append(float(E[i] @ E[RNG.choice(other)].T))
print(f"  similarite cosine moyenne, meme style dominant : {np.mean(sims_intra):.3f}")
print(f"  similarite cosine moyenne, styles differents    : {np.mean(sims_inter):.3f}")
print("  => le contenu porte bien le style, et il le porte pour TOUS les items, froids compris.")

plongement de contenu : 2000 items x 32 dimensions
variance expliquee cumulee : 0.438

Controle : les items partageant le style dominant se ressemblent-ils dans l'espace reduit ?
  similarite cosine moyenne, meme style dominant : 0.474
  similarite cosine moyenne, styles differents    : 0.111
  => le contenu porte bien le style, et il le porte pour TOUS les items, froids compris.


Le contrôle ci-dessus n'est pas décoratif : il établit que l'information de style survit à la réduction de dimension. Sans lui, un mauvais résultat des modèles de contenu serait imputable au plongement, et la comparaison avec les identifiants sémantiques n'aurait aucun sens — les deux modèles reposeraient sur une représentation déjà dégradée.

## Les identifiants sémantiques

La quantification résiduelle transforme un vecteur continu en une **suite de codes** : on découpe l'espace en `K` cellules, on code chaque item par sa cellule, puis on quantifie *le résidu* — l'écart au centre de sa cellule — par un second dictionnaire, et ainsi de suite. Trois niveaux de `K` codes donnent un identifiant de `3 log2 K` bits. Un item devient donc un tuple, et deux items qui partagent les premiers codes sont « voisins de haut niveau ».

C'est le mécanisme du rapport Sona, réduit à sa mécanique : un plongement, des dictionnaires résiduels, des tuples. Rien de propriétaire dans l'opération.

In [5]:
def residual_quantize(embeddings, n_levels=3, codebook=32, seed=42):
    """Quantification residuelle : renvoie (codes (n, n_levels), dictionnaires)."""
    residual = embeddings.copy()
    codes, books = [], []
    for level in range(n_levels):
        km = KMeans(n_clusters=codebook, n_init=4, random_state=seed + level)
        labels = km.fit_predict(residual)
        codes.append(labels)
        books.append(km.cluster_centers_)
        residual = residual - km.cluster_centers_[labels]
    return np.stack(codes, axis=1), books

N_LEVELS, CODEBOOK = 3, 32
sid_codes, sid_books = residual_quantize(E, N_LEVELS, CODEBOOK)
bits_per_item = N_LEVELS * math.log2(CODEBOOK)

reconstruction = np.zeros_like(E)
residual = E.copy()
for level in range(N_LEVELS):
    reconstruction += sid_books[level][sid_codes[:, level]]
    residual = E - reconstruction

print(f"identifiants : {N_LEVELS} niveaux x {CODEBOOK} codes = {bits_per_item:.1f} bits par item")
print(f"(un item brut couterait {E.shape[1] * 32} bits en float32)")
print()
print("Quatre identifiants tires au hasard, avec le style dominant reel de l'item :")
for i in RNG.choice(N_ITEMS, size=4, replace=False):
    print(f"  item {i:4d} | id {tuple(int(c) for c in sid_codes[i])} | style {STYLE_NAMES[item_primary[i]]:>9}")
print()
print(f"erreur de quantification residuelle : {np.linalg.norm(E - reconstruction, axis=1).mean():.4f} "
      f"(norme initiale moyenne : {np.linalg.norm(E, axis=1).mean():.4f})")
print(f"codes de premier niveau distincts couverts : {len(set(sid_codes[:, 0]))} / {CODEBOOK}")

identifiants : 3 niveaux x 32 codes = 15.0 bits par item
(un item brut couterait 1024 bits en float32)

Quatre identifiants tires au hasard, avec le style dominant reel de l'item :
  item 1313 | id (8, 3, 13) | style    reggae
  item 1315 | id (5, 18, 6) | style      folk
  item 1945 | id (30, 21, 19) | style     world
  item   95 | id (5, 18, 28) | style      folk

erreur de quantification residuelle : 0.3581 (norme initiale moyenne : 1.0000)
codes de premier niveau distincts couverts : 32 / 32


Deux propriétés de la sortie comptent pour la suite. D'abord, la quantification **est** une compression avec perte : l'erreur résiduelle moyenne n'est pas nulle, et c'est son prix qu'il faudra mettre en regard de ce qu'elle rapporte. Ensuite, le premier niveau joue le rôle d'un **regroupement grossier** : deux items qui partagent leur premier code sont traités comme proches parentes par tout modèle qui range dans cet espace, y compris s'ils ne se ressemblent pas finement.

## Les quatre lectures d'un même catalogue

Les modèles comparés ne sont pas quatre variantes d'une même idée, mais quatre réponses à la question « d'où vient le signal ? » :

| Modèle | Signal utilisé | Comportement attendu sur un item froid |
|---|---|---|
| **Populaire** | comptage global des interactions | nulle part — un item jamais vu compte zéro |
| **Collaboratif** (SVD tronquée sur la matrice binaire) | co-occurrences d'interactions | aucune coordonnée — la colonne de l'item est vide |
| **Contenu** (cosinus au profil) | étiquettes de l'item contre étiquettes consommées | pleine capacité — le contenu existe sans interaction |
| **Identifiants sémantiques** | codes quantifiés contre distribution de codes consommés | pleine capacité, mais à travers la compression |

La baseline de contenu est délibérément **forte** : c'est elle qui rend le résultat des identifiants sémantiques interprétable. Si les identifiants ne battaient qu'un modèle collaboratif aveugle aux items froids, on n'aurait rien appris sur la quantification — seulement sur le démarrage à froid.

In [6]:
n_users = N_USERS
train_user = train_pairs_final[:, 0]
train_item = train_pairs_final[:, 1]

# matrice binaire d'entrainement
X = np.zeros((n_users, N_ITEMS), dtype=np.float32)
X[train_user, train_item] = 1.0
seen_items = {u: set(train_item[train_user == u].tolist()) for u in range(n_users)}

# --- 1. populaire
counts = X.sum(axis=0)
S_pop = np.tile(counts, (n_users, 1))

# --- 2. collaboratif : SVD tronquee sur la matrice binaire
svd_col = TruncatedSVD(n_components=32, random_state=42)
U_col = svd_col.fit_transform(X)          # U * S
V_col = svd_col.components_.T             # vecteurs d'items
S_col = U_col @ V_col.T

# --- 3. contenu : profil = moyenne des plongements consommes
P = np.zeros((n_users, EMB_DIM))
for u in range(n_users):
    idx = np.fromiter(seen_items[u], dtype=int)
    if len(idx):
        P[u] = E[idx].mean(axis=0)
P /= np.maximum(np.linalg.norm(P, axis=1, keepdims=True), 1e-12)
S_content = P @ E.T

# --- 4. identifiants semantiques : distribution de codes par utilisateur, en PMI
ALPHA = 0.5
prior = []
for level in range(N_LEVELS):
    c = np.bincount(sid_codes[:, level], minlength=CODEBOOK).astype(float)
    prior.append((c + 1.0) / (c.sum() + CODEBOOK))
log_prior = [np.log(p) for p in prior]

S_sid = np.zeros((n_users, N_ITEMS))
for u in range(n_users):
    idx = np.fromiter(seen_items[u], dtype=int)
    if not len(idx):
        continue
    score = np.zeros(N_ITEMS)
    for level in range(N_LEVELS):
        c = np.bincount(sid_codes[idx, level], minlength=CODEBOOK).astype(float)
        p = (c + ALPHA * prior[level] * CODEBOOK) / (len(idx) + ALPHA * CODEBOOK)
        score += np.log(p)[sid_codes[:, level]] - log_prior[level][sid_codes[:, level]]
    S_sid[u] = score

MODELS = {"Populaire": S_pop, "Collaboratif (SVD)": S_col, "Contenu (cosinus)": S_content, "Identifiants semantiques": S_sid}
print("quatre matrices de score construites :", {k: v.shape for k, v in MODELS.items()})
print(f"items froids : ils n'ont AUCUNE colonne non nulle dans la matrice d'entrainement "
      f"({int((X[:, is_cold].sum(axis=0) == 0).sum())} / {n_items_cold} colonnes vides)")

quatre matrices de score construites : {'Populaire': (800, 2000), 'Collaboratif (SVD)': (800, 2000), 'Contenu (cosinus)': (800, 2000), 'Identifiants semantiques': (800, 2000)}
items froids : ils n'ont AUCUNE colonne non nulle dans la matrice d'entrainement (300 / 300 colonnes vides)


La dernière ligne de la sortie est le mécanisme du problème, vérifié plutôt que raconté : les colonnes des items froids sont vides, donc le modèle collaboratif ne peut leur attribuer qu'un score constant. La cellule suivante mesure les conséquences, et elle mesure aussi ce qu'un classement au hasard donnerait, pour que les chiffres soient lus contre une référence et non dans le vide.

In [7]:
K = 20

def rank_metrics(scores, targets, seen, k=K):
    """Recall@k et NDCG@k moyens : la cible tombe-t-elle dans le top-k des items non consommes ?"""
    recalls, ndcgs = [], []
    for u, target in targets.items():
        s = scores[u].copy()
        idx = np.fromiter(seen[u], dtype=int)
        s[idx] = -np.inf
        order = np.argsort(-s, kind="stable")[:k]
        if target in order:
            recalls.append(1.0)
            rank = int(np.where(order == target)[0][0]) + 1
            ndcgs.append(1.0 / math.log2(rank + 1))
        else:
            recalls.append(0.0)
            ndcgs.append(0.0)
    return float(np.mean(recalls)), float(np.mean(ndcgs))

rows = []
for name, S in MODELS.items():
    r_warm, n_warm = rank_metrics(S, warm_target, seen_items)
    r_cold, n_cold = rank_metrics(S, cold_target, seen_items)
    rows.append({"Modele": name, "Recall@20 chaud": r_warm, "NDCG@20 chaud": n_warm,
                 "Recall@20 froid": r_cold, "NDCG@20 froid": n_cold})
results = pd.DataFrame(rows).set_index("Modele")

# reference analytique du hasard : k / (taille de l'univers de candidats)
cand = N_ITEMS - np.mean([len(v) for v in seen_items.values()])
print(f"hasard : Recall@{K} attendu = {K / cand:.4f}  (univers moyen de {cand:.0f} candidats)")
print()
print((results * 100).round(2).to_string())
print()
print("en points de pourcentage de Recall@20 :")
for name in results.index:
    print(f"  {name:<28} chaud {results.loc[name, 'Recall@20 chaud']*100:5.2f}  "
          f"froid {results.loc[name, 'Recall@20 froid']*100:5.2f}")

hasard : Recall@20 attendu = 0.0102  (univers moyen de 1967 candidats)

                          Recall@20 chaud  NDCG@20 chaud  Recall@20 froid  NDCG@20 froid
Modele                                                                                  
Populaire                            4.38           1.58             0.00           0.00
Collaboratif (SVD)                   3.38           1.37             0.00           0.00
Contenu (cosinus)                    1.75           0.50             1.88           0.53
Identifiants semantiques             2.25           0.71             1.63           0.66

en points de pourcentage de Recall@20 :
  Populaire                    chaud  4.38  froid  0.00
  Collaboratif (SVD)           chaud  3.38  froid  0.00
  Contenu (cosinus)            chaud  1.75  froid  1.88
  Identifiants semantiques     chaud  2.25  froid  1.63


## Les écarts sont-ils plus grands que le bruit ?

Un rappel de 1,88 % contre 1,63 % ne veut rien dire tant qu'on ne sait pas de combien deux modèles **identiques** varieraient sur le même échantillon. Les deux contrastes qui portent la conclusion sont donc testés appariés — mêmes utilisateurs, mêmes cibles, seule change la matrice de score.

Deux instruments, complémentaires :

- le **test exact de McNemar** sur les paires discordantes (là où un modèle atteint la cible et l'autre non). Il ne suppose rien sur la forme de la distribution des rappels, seulement que les utilisateurs sont indépendants ;
- l'**intervalle de confiance bootstrap** de la différence de rappel, qui donne l'ordre de grandeur en points de pourcentage — la grandeur que le lecteur compare intuitivement.


In [8]:
from scipy.stats import binomtest

def hit_vector(S, targets, seen, k=K):
    """Vecteur de succes dans le meme ordre de cibles, pour deux modeles comparables."""
    hits = []
    for u, target in targets.items():
        s = S[u].copy()
        s[np.fromiter(seen[u], dtype=int)] = -np.inf
        order = np.argsort(-s, kind="stable")[:k]
        hits.append(target in order)
    return np.array(hits, dtype=bool)

def bootstrap_diff(hits_a, hits_b, n_boot=4000, seed=7):
    """IC 95 % de la difference de rappel (a - b), reechantillonnage des utilisateurs."""
    rng = np.random.default_rng(seed)
    n = len(hits_a)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[b] = hits_a[idx].mean() - hits_b[idx].mean()
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

contrasts = [
    ("Identifiants vs Contenu (froid)",  "Identifiants semantiques", "Contenu (cosinus)", cold_target),
    ("Identifiants vs Contenu (chaud)",  "Identifiants semantiques", "Contenu (cosinus)", warm_target),
    ("Populaire vs Contenu (chaud)",     "Populaire",                "Contenu (cosinus)", warm_target),
    ("Collaboratif vs Populaire (chaud)", "Collaboratif (SVD)",      "Populaire",         warm_target),
]

print(f"{'contraste':<38} {'A':>7} {'B':>7} {'ecart':>10} {'IC 95 %':>19} {'McNemar p':>10}")
for label, a_name, b_name, targets in contrasts:
    ha = hit_vector(MODELS[a_name], targets, seen_items)
    hb = hit_vector(MODELS[b_name], targets, seen_items)
    lo, hi = bootstrap_diff(ha, hb)
    both = ha & hb
    a_only = int((ha & ~hb).sum())
    b_only = int((hb & ~ha).sum())
    p = 1.0 if a_only + b_only == 0 else binomtest(max(a_only, b_only), a_only + b_only, 0.5).pvalue
    sign = "significatif" if p < 0.05 else "non concluant"
    print(f"{label:<38} {ha.mean()*100:5.2f}% {hb.mean()*100:5.2f}% "
          f"{(ha.mean()-hb.mean())*100:+8.2f} pts  [{lo*100:+5.2f} ; {hi*100:+5.2f}] {p:10.4f}  {sign}")


contraste                                    A       B      ecart             IC 95 %  McNemar p


Identifiants vs Contenu (froid)         1.63%  1.88%    -0.25 pts  [-1.38 ; +0.88]     0.8318  non concluant


Identifiants vs Contenu (chaud)         2.25%  1.75%    +0.50 pts  [-0.75 ; +1.75]     0.5716  non concluant


Populaire vs Contenu (chaud)            4.38%  1.75%    +2.62 pts  [+1.00 ; +4.25]     0.0038  significatif


Collaboratif vs Populaire (chaud)       3.38%  4.38%    -1.00 pts  [-2.62 ; +0.75]     0.3123  non concluant


## Ce que les tests autorisent à écrire

Sur quatre contrastes, **un seul est établi**. Le populaire bat le contenu direct sur les items chauds de 2,62 points de rappel, intervalle de confiance [+1,00 ; +4,25], test de McNemar à p = 0,0038 — un effet qui survit au bruit et que le protocole génératif du corpus explique (la popularité y est corrélée à l'affinité, par construction).

Les trois autres contrastes sont **non concluants**, et il faut le dire sans détour :

- identifiants contre contenu direct, sur le froid : −0,25 point, intervalle [−1,38 ; +0,88], p = 0,83. La différence observable est du même ordre que ce que deux tirages d'utilisateurs produiraient ;
- identifiants contre contenu direct, sur le chaud : +0,50 point, intervalle [−0,75 ; +1,75], p = 0,57 ;
- collaboratif contre populaire, sur le chaud : −1,00 point, intervalle [−2,62 ; +0,75], p = 0,31.

La leçon méthodologique vaut plus que les chiffres : avec huit cents utilisateurs, un rappel autour de deux pour cent et vingt candidats retenus, il faut un écart de l'ordre de **deux points et demi** pour que McNemar le distingue du hasard. Toute conclusion tirée d'un écart d'un demi-point sur cet échantillon aurait été une lecture de bruit — et c'est précisément le piège que ce genre de comparaison tend au lecteur pressé.

Deux effets restent **structurels**, donc vrais indépendamment de l'échantillon : les colonnes vides des items froids rendent leur score constant pour le populaire et le collaboratif (zéro rappel, et ce zéro n'est pas une estimation mais une impossibilité), et le contenu, lui, voit tous les items. Ce qui n'est pas établi, c'est que la **quantification** ajoute quoi que ce soit à ce que le contenu direct apporte déjà.


Deux lectures se dégagent, et elles ne vont pas dans le même sens — c'est le résultat du carnet.

**Sur les items froids**, le classement bascule : le modèle populaire et le modèle collaboratif rendent exactement **zéro**, pour une raison structurelle et non statistique — leurs colonnes sont vides, ils ne peuvent pas distinguer un item froid d'un autre. Les deux modèles qui voient le contenu atteignent la cible dans environ deux cas sur cent, contre une référence de hasard à 1,02 % : le contenu apporte donc un signal réel sur une population où le collaboratif est aveugle par construction.

**Sur les items chauds**, l'ordre s'inverse partiellement : le populaire domine, le collaboratif suit, les identifiants sémantiques devancent le contenu direct, et le contenu direct arrive en dernier. La domination du populaire n'est pas un défaut de méthode : dans le protocole génératif de ce corpus, la probabilité d'interaction est le produit de l'affinité **et** d'une popularité log-normale, si bien que le comptage global encode une partie de la préférence. Un corpus à popularité uniforme renverserait ce classement — c'est une propriété du corpus, pas une loi.

Le point qui compte pour la question posée est ailleurs, et il est plus sobre : **les identifiants sémantiques ne battent pas le contenu direct sur le froid**, alors qu'ils le battent légèrement sur le chaud. La quantification n'est donc pas un gain gratuit sur le démarrage à froid ; elle redistribue la qualité entre les deux populations. Encore faut-il savoir si l'écart froid survit au bruit — c'est l'objet de la section suivante.


## Le prix de la compression

Ranger dans un espace d'identifiants n'est pas gratuit : la quantification détruit de l'information, et le nombre de codes par niveau règle directement cette perte. La cellule suivante fait varier le dictionnaire à nombre de niveaux fixé, et reporte le coût en bits par item en regard du rappel obtenu sur les items froids. Le but n'est pas de trouver « le bon » réglage, mais de voir la **forme de la courbe** : si le rappel est plat sur une large plage de tailles, la compression est bon marché ; s'il s'effondre, l'identifiant sémantique n'est utilisable qu'à un coût qu'il faut alors nommer.

In [9]:
sweep = []
for cb in (8, 16, 32, 64):
    codes_cb, _ = residual_quantize(E, N_LEVELS, cb)
    S = np.zeros((n_users, N_ITEMS))
    prior_cb = []
    for level in range(N_LEVELS):
        c = np.bincount(codes_cb[:, level], minlength=cb).astype(float)
        prior_cb.append((c + 1.0) / (c.sum() + cb))
    for u in range(n_users):
        idx = np.fromiter(seen_items[u], dtype=int)
        if not len(idx):
            continue
        score = np.zeros(N_ITEMS)
        for level in range(N_LEVELS):
            c = np.bincount(codes_cb[idx, level], minlength=cb).astype(float)
            p = (c + ALPHA * prior_cb[level] * cb) / (len(idx) + ALPHA * cb)
            score += np.log(p)[codes_cb[:, level]] - np.log(prior_cb[level])[codes_cb[:, level]]
        S[u] = score
    r_cold, n_cold = rank_metrics(S, cold_target, seen_items)
    r_warm, _ = rank_metrics(S, warm_target, seen_items)
    sweep.append({"codes par niveau": cb, "bits par item": N_LEVELS * math.log2(cb),
                  "Recall@20 froid": r_cold, "Recall@20 chaud": r_warm, "NDCG@20 froid": n_cold})

sweep_df = pd.DataFrame(sweep).set_index("codes par niveau")
print(sweep_df.assign(**{c: (sweep_df[c] * 100).round(2) for c in sweep_df.columns if c != "bits par item"}).to_string())
print()
best = sweep_df["Recall@20 froid"].idxmax()
print(f"meilleur rappel froid au dictionnaire de {best} codes "
      f"({N_LEVELS * math.log2(best):.1f} bits par item)")

                  bits par item  Recall@20 froid  Recall@20 chaud  NDCG@20 froid
codes par niveau                                                                
8                           9.0             1.38             1.12           0.61
16                         12.0             1.75             2.00           0.69
32                         15.0             1.63             2.25           0.66
64                         18.0             1.50             1.62           0.56

meilleur rappel froid au dictionnaire de 16 codes (12.0 bits par item)


La courbe n'a pas la forme attendue : le rappel froid ne croît pas avec la taille du dictionnaire. Il culmine au dictionnaire de 16 codes (12 bits par item), redescend à 32 puis à 64, et le dictionnaire le plus étroit (8 codes, 9 bits) est le plus faible. Une compression très agressive coûte donc visiblement, mais au-delà de 12 bits, ajouter des codes n'achète rien — et peut même coûter, ce qui est le signe d'un plafond d'estimation plutôt que d'un vrai retournement.

Ce plafond est cohérent avec la taille de l'échantillon : la mesure porte sur les utilisateurs qui ont une cible froide, et l'écart entre deux réglages voisins du balayage vaut une fraction de point. **Ce balayage ne permet donc pas de choisir un réglage** ; il établit seulement que le coût de la compression n'est pas monotone à cette échelle, et il montre la bonne façon de poser la question — en bits par item, pas en nombre de codes.

La conséquence pratique est celle qu'un ingénieur retiendrait : la quantification résiduelle est un levier de **coût de stockage** et de **partage de préfixe**, pas un levier de qualité à lui seul. Ranger dans un espace d'identifiants ne rend pas un catalogue plus recommandable ; cela rend le catalogue **générable** — on peut produire un identifiant, pas une matrice de scores dense — et c'est ce que le rapport Sona exploite en production pour unifier génération de candidats et ranking.


## Ce que ce carnet établit, et ce qu'il ne dit pas

**Établi par la mesure** — le mécanisme du démarrage à froid, vérifié plutôt que raconté : les colonnes des items froids sont vides, donc le comptage populaire et la décomposition collaborative leur attribuent un score constant et un rappel exactement nul, tandis que les deux modèles qui lisent le contenu atteignent la cible dans près de deux cas sur cent contre une référence de hasard à 1,02 %. Établi aussi, et c'est le résultat le plus utile : un écart d'un demi-point de rappel sur cet échantillon **ne se distingue pas du bruit**, ce que les tests appariés montrent noir sur blanc — seule la domination du populaire sur le contenu chez les items chauds franchit le seuil.

**Non établi** — que ranger dans un espace d'identifiants quantifiés vaille mieux, ou moins bien, que ranger directement dans l'espace de contenu. Les deux populations donnent des écarts de signe opposé, tous deux non concluants. La quantification coûte de l'information (erreur résiduelle mesurée au-dessus) ; ce qu'elle rapporte n'apparaît pas à cette échelle, et le balayage des tailles de dictionnaire ne montre pas de tendance monotone qui sauverait la thèse.

**Ce qu'il serait malhonnête de suggérer** :

- rien ici ne dit ce que vaut un recommandeur génératif **entraîné** : le rapport Sona apprend l'encodeur par un objectif contrastif collaboratif et un décodeur autorégressif sur les identifiants. Ce carnet garde le plongement figé et ne fait que **ranger** ;
- le corpus est contrôlé, donc les valeurs absolues ne se transportent pas sur un catalogue réel ; seuls les effets structurels et les écarts testés s'interprètent ;
- la compression d'historique du papier n'est pas reproduite : le profil utilisateur est une moyenne, pas une mémoire hiérarchique.

**Ce que la quantification apporte réellement**, et que ce carnet donne à voir sans le surinterpréter : un item devient une **suite de codes** — quinze bits au lieu de mille vingt-quatre — donc quelque chose qu'un modèle peut *produire*, et non seulement *scorer*. C'est un changement de nature de la sortie, pas un gain de rappel démontré ici. Un lecteur qui retiendrait « les identifiants sémantiques recommandent mieux » aurait retenu l'inverse de ce que les tests disent.


## Exercices

Les trois exercices suivants prolongent les mesures ci-dessus. Aucun n'est résolu dans le carnet.

In [10]:
# Exercice 1 : deux niveaux suffisent-ils ?
# TODO etudiant : quantifie E en 2 niveaux x 32 codes, construis le modele d'identifiants
# correspondant, et compare Recall@20 froid a celui obtenu avec 3 niveaux.
# Indice : reutilise residual_quantize(E, n_levels=2, codebook=32) et la boucle de score PMI.
# Etape 1 : codes2, _ = residual_quantize(E, 2, 32)
# Etape 2 : reconstruis S_sid avec ces codes (meme formule de lissage)
# Etape 3 : rank_metrics(S_2niveaux, cold_target, seen_items)
result_ex1 = None  # TODO etudiant
print("Exercice 1 a completer")

Exercice 1 a completer


In [11]:
# Exercice 2 : le partage du premier code aide-t-il vraiment ?
# TODO etudiant : parmi les items froids atteints par le modele d'identifiants, mesure la part
# dont le premier code est partage avec un item que l'utilisateur a consomme.
# Indice : compare sid_codes[target, 0] aux codes de premier niveau des items de seen_items[u].
# Etape 1 : pour chaque utilisateur de cold_target, teste si la cible est dans le top-20
# Etape 2 : parmi les succes, compte ceux ou sid_codes[cible, 0] est dans les codes consommes
# Etape 3 : compare cette part a ce que donnerait un appariement aleatoire de premier niveau
result_ex2 = None  # TODO etudiant
print("Exercice 2 a completer")

Exercice 2 a completer


In [12]:
# Exercice 3 : que se passe-t-il si le contenu est moins informatif ?
# TODO etudiant : bruite une fraction des etiquettes (remplace un mot sur trois par un mot
# tire au hasard dans le vocabulaire), recalcule plongement et modeles, et mesure la chute du
# rappel froid pour le contenu et pour les identifiants semantiques.
# Indice : la degradation devrait toucher les deux modeles de contenu ; l'interet est de
# comparer LAQUELLE resiste le mieux au meme bruit.
# Etape 1 : construis une liste metadata_bruitee (fraction configurable)
# Etape 2 : recalcule E_bruite puis les deux matrices de score
# Etape 3 : compare les rappels froids a ceux de la cellule de resultats
result_ex3 = None  # TODO etudiant
print("Exercice 3 a completer")

Exercice 3 a completer


## Références

- **Sona** — équipe Yandex Music, *Sona Technical Report*, [arXiv:2608.11015](https://arxiv.org/abs/2608.11015) : recommandeur génératif mono-modèle, tokenisation sémantique par quantification résiduelle, compression d'historique. Cité pour le paradigme et le vocabulaire ; aucune valeur numérique n'en est reprise.
- [ML-07-Recommendation-Python](ML-07-Recommendation-Python.ipynb) — la factorisation matricielle dont ce carnet mesure la limite froide.
- [ML-08-Clustering-Python](ML-08-Clustering-Python.ipynb) — le précédent du corpus contrôlé à vérité terrain cachée.